Resume PDF → Text Extraction → Preprocessing →
→ Representation (TF-IDF + BERT) → Similarity →
→ Skill Extraction → Missing Skills → Output

# Step 1 : Extract text from resume

In [ ]:
!pip install pdfplumber  #pdfplumber library to extract raw textual content from resumes for downstream NLP processing

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 98.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 111.8 MB/s eta 0:00:00


In [ ]:
import pdfplumber

def extract_text_from_pdf(file):
  text=""
  with pdfplumber.open(file) as pdf:
    for page in pdf.pages:
      text += page.extract_text() or ""
  return text

# Step2 : Text Preprocessing

In [ ]:
import nltk , re
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
stop_words = set(stopwords.words('english'))
stop_words

{'a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 "he's",
 'her',
 'here',
 'hers',
 'herself',
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 "i'll",
 "i'm",
 "i've",
 'if',
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [ ]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text) # remove everything except alphanumerics

    tokens = word_tokenize(text)

    tokens = [word for word in tokens if word not in stop_words]

    return " ".join(tokens)

# Step 4 : Cosine Similarity
( angle between vectors
Same meaning → small angle → high score  
Different → large angle → low score)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def tfidf_similarity(resume, job):
    vectorizer = TfidfVectorizer()
    tfidf = vectorizer.fit_transform([resume, job])

    score = cosine_similarity(tfidf[0:1], tfidf[1:2])
    return float(score[0][0]) * 100

# Step 5 : BERT
sentence BERT ---> Better sentence similarity using cosine similarity

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer , util
model = SentenceTransformer('all-MiniLM-L6-v2')

def bert_similarity(resume,job_desc):
  emb1 = model.encode(resume,convert_to_tensor=True)  # embeedings
  emb2 = model.encode(job_desc,convert_to_tensor=True)

  score = util.cos_sim(emb1,emb2)
  return float(score[0][0])*100

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Step 6 : Combine both scores
hybrid scoring mechanism



more weight to bert

In [ ]:
def final_score(tfidf_score,bert_score):
  return 0.3*tfidf_score + 0.7*bert_score

# Step 7 : Skill Extraction

1 ) Categorized skill Dictionary

In [ ]:
skills_dict = {
    "programming": [
        "python", "java", "c++", "javascript", "c", "go", "rust"
    ],

    "data": [
        "sql", "excel", "power bi", "tableau", "data analysis"
    ],

    "aiml": [
        "machine learning", "deep learning", "nlp",
        "computer vision", "transformers"
    ],

    "web": [
        "html", "css", "react", "node", "express", "django", "flask"
    ],

    "cloud_devops": [
        "aws", "azure", "gcp", "docker", "kubernetes", "ci/cd"
    ],

    "tools": [
        "git", "github", "linux", "bash", "jira"
    ]
}

2: Flatten it (for use in code)

In [ ]:
def get_all_skills(skills_dict):
    all_skills = []
    for category in skills_dict.values():
        all_skills.extend(category)
    return all_skills

skills_list = get_all_skills(skills_dict)

3: Use Phrase-Based Matching

In [ ]:
def extract_skills_from_text(text, skills_list):
    text = text.lower()
    found = []

    for skill in skills_list:
        if skill in text:
            found.append(skill)

    return list(set(found))

4 : Show Category-wise Gap




In [ ]:
def categorize_missing_skills(missing_skills, skills_dict):
    category_gap = {}

    for category, skills in skills_dict.items():
        gap = [skill for skill in missing_skills if skill in skills]
        if gap:
            category_gap[category] = gap

    return category_gap

# FULL PIPELINE TEST

In [ ]:
# ===== INPUT =====
resume_text = extract_text_from_pdf('resume (1).pdf')

job_desc = """Looking for a machine learning engineer with experience in Python, NLP, and deep learning."""

# ===== PREPROCESS (ONLY FOR TF-IDF) =====
resume_clean = preprocess(resume_text)
job_clean = preprocess(job_desc)

# ===== TF-IDF SCORE =====
tfidf_score = tfidf_similarity(resume_clean, job_clean)

# ===== BERT SCORE =====
bert_score = bert_similarity(resume_text, job_desc)

# ===== FINAL SCORE =====
score = final_score(tfidf_score, bert_score)

# ===== SKILL EXTRACTION =====
resume_skills = extract_skills_from_text(resume_text, skills_list)
job_skills = extract_skills_from_text(job_desc, skills_list)

# ===== MISSING SKILLS =====
missing = list(set(job_skills) - set(resume_skills))

# ===== CATEGORY GAP =====
category_gap = categorize_missing_skills(missing, skills_dict)

# ===== OUTPUT =====
print("Final Score:", round(score, 2))
print("TF-IDF Score:", round(tfidf_score, 2))
print("BERT Score:", round(bert_score, 2))

print("\nResume Skills:", resume_skills)
print("Job Skills:", job_skills)

print("\nMissing Skills:", missing)
print("Category Gap:", category_gap)

Final Score: 42.51
TF-IDF Score: 31.89
BERT Score: 47.06

Resume Skills: ['sql', 'github', 'data analysis', 'machine learning', 'transformers', 'go', 'git', 'c++', 'python', 'deep learning', 'c']
Job Skills: ['machine learning', 'nlp', 'python', 'deep learning', 'c']

Missing Skills: ['nlp']
Category Gap: {'aiml': ['nlp']}


Built:

Information Retrieval System


 NLP Similarity Engine


 Skill Gap Analyzer


 Hybrid Model (TF-IDF + BERT)

# Create app.py

In [ ]:
code = """
import streamlit as st
import pdfplumber
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer, util

# ===== SETUP =====
nltk.download('punkt')
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# ===== FUNCTIONS =====
def extract_text_from_pdf(file):
    text = ""
    with pdfplumber.open(file) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""
    return text

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\\s]', '', text) # Corrected regex and escaped backslash
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    return " ".join(tokens)

def tfidf_similarity(resume, job):
    vectorizer = TfidfVectorizer()
    tfidf = vectorizer.fit_transform([resume, job])
    score = cosine_similarity(tfidf[0:1], tfidf[1:2])
    return float(score[0][0]) * 100

model = SentenceTransformer('all-MiniLM-L6-v2')

def bert_similarity(resume, job):
    emb1 = model.encode(resume, convert_to_tensor=True)
    emb2 = model.encode(job, convert_to_tensor=True)
    return float(util.cos_sim(emb1, emb2)[0][0]) * 100

def skill_match_score(resume_skills, job_skills):
    if len(job_skills) == 0:
        return 0
    return (len(set(resume_skills) & set(job_skills)) / len(job_skills)) * 100

def final_score(tfidf_score, bert_score, skill_score):
    return 0.3 * tfidf_score + 0.5 * bert_score + 0.2 * skill_score

skills_dict = {
    "programming": ["python", "java", "c++", "javascript"],
    "data": ["sql", "excel", "tableau", "power bi"],
    "aiml": ["machine learning", "deep learning", "nlp"],
    "web": ["html", "css", "react", "node"],
    "cloud": ["aws", "docker", "kubernetes"]
}

def get_all_skills(skills_dict):
    skills = []
    for cat in skills_dict.values():
        skills.extend(cat)
    return skills

skills_list = get_all_skills(skills_dict)

def extract_skills_from_text(text, skills_list):
    text = text.lower()
    found = []
    for skill in skills_list:
        if skill in text:
            found.append(skill)
    return list(set(found))

def categorize_missing_skills(missing, skills_dict):
    result = {}
    for cat, skills in skills_dict.items():
        gap = [s for s in missing if s in skills]
        if gap:
            result[cat] = gap
    return result

# ===== UI =====
st.title("AI Resume Analyzer")

resume_file = st.file_uploader("Upload Resume (PDF)", type=["pdf"])
job_desc = st.text_area("Paste Job Description")

if st.button("Analyze"):
    if resume_file and job_desc:

        resume_text = extract_text_from_pdf(resume_file)

        resume_clean = preprocess(resume_text)
        job_clean = preprocess(job_desc)

        tfidf_score = tfidf_similarity(resume_clean, job_clean)
        bert_score = bert_similarity(resume_text, job_desc)

        resume_skills = extract_skills_from_text(resume_text, skills_list)
        job_skills = extract_skills_from_text(job_desc, skills_list)

        missing = list(set(job_skills) - set(resume_skills))
        category_gap = categorize_missing_skills(missing, skills_dict)

        skill_score = skill_match_score(resume_skills, job_skills)

        score = final_score(tfidf_score, bert_score, skill_score)

        st.subheader("Match Score")
        st.progress(int(score))
        st.write(f"{score:.2f}% Match")

        st.write("Your Skills:", resume_skills)
        st.write("Missing Skills:", missing)
        st.write("Skill Gap:", category_gap)

        st.write("TF-IDF:", tfidf_score)
        st.write("BERT:", bert_score)
        st.write("Skill Match:", skill_score)

    else:
        st.error("Upload resume and job description")
"""

with open("app.py", "w") as f:
    f.write(code)


In [ ]:
!pip install streamlit pyngrok pdfplumber sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 48.3 MB/s eta 0:00:00


In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("3BzvUM00Le5gh3RSjc0ckvaAXj8_88BfMfkedAndKaHEWWkz4")

In [ ]:
public_url = ngrok.connect(8501)
print(public_url)

!streamlit run app.py &

NgrokTunnel: "https://hirundine-leonidas-unattenuated.ngrok-free.dev" -> "http://localhost:8501"



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.142.183.92:8501

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
Loading weights: 100% 103/103 [00:00<00:00, 3088.63it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package 